# Chanakya-Repair fine-tune (Unsloth LoRA on Qwen2.5)

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then upload `repair-dataset.jsonl` using the folder panel on the left.

Run every cell top to bottom. Training a few hundred examples takes ~10-20 min on a T4.

## 1. Install Unsloth

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. Load the base model in 4-bit

Use the 3B model if you hit out-of-memory, or for faster CPU inference later.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # or ...-3B-Instruct-bnb-4bit
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
)

## 3. Attach LoRA adapters (only ~1% of weights get trained)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Load the dataset and apply the Qwen chat template

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
dataset = load_dataset("json", data_files="repair-dataset.jsonl", split="train")

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False,
                                                   add_generation_prompt=False)
                     for m in batch["messages"]]}

dataset = dataset.map(to_text, batched=True)
print(f"Loaded {len(dataset)} training examples")

## 5. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        bf16 = torch.cuda.is_bf16_supported(),
        fp16 = not torch.cuda.is_bf16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        logging_steps = 1,
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer.train()

## 6. Export to GGUF for Ollama

When done, open the `chanakya-repair-gguf/` folder on the left, download the `.gguf` file, put the `Modelfile` next to it, and run `ollama create chanakya-repair -f Modelfile` on your PC.

In [ ]:
model.save_pretrained_gguf("chanakya-repair-gguf", tokenizer,
                           quantization_method="q4_k_m")
print("Done. Download the .gguf from the chanakya-repair-gguf/ folder.")